In [1]:
# Installing required verions of sympy
!pip uninstall -y sympy
!pip install sympy==1.13.1

Found existing installation: sympy 1.14.0
Uninstalling sympy-1.14.0:
  Successfully uninstalled sympy-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 34.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires sympy>=1.13.3, but you have sympy 1.13.1 which is incompatible.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np

In [5]:
device = "cuda"
model_name = "facebook/esm2_t30_150M_UR50D"               #ESM2 150M  my meta -- Proteain language Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print('Model Sucessfully Loaded')

Loading weights:   0%|          | 0/486 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Sucessfully Loaded


In [10]:
proteases = pd.read_csv('/content/uniprotkb_ec_3_4_21_AND_length_450_TO_5_2026_04_14 (1).tsv',sep = '\t') # Loading data
embedding_vec = []          # List to store embedding
for i in range(len(proteases)):
  sequence = proteases.iloc[i]['Sequence']
  inputs = tokenizer(sequence, return_tensors="pt").to(device)
  with torch.no_grad():
      outputs = model(**inputs)
  embeddings = outputs.last_hidden_state[:, 1:-1] # Extracting Last layer embdding. CLS token can also be used
  sequence_embedding = embeddings.mean(dim=1)     # Averaging it according to sequence size
  sequence_embedding = sequence_embedding.cpu().detach().numpy()    # Converting to procesable form
  embedding_vec.append(sequence_embedding)
proteases['Encoding'] = embedding_vec   # Storing the computed encoding in the list


In [11]:
hip_1 = proteases.iloc[21]['Encoding'][0]   # Taking the embeddings of Hip !
sim = []      # List to store the similarity score
for i in range(len(proteases)):
  enc = proteases.iloc[i]['Encoding'][0]
  cos_sim = np.dot(hip_1,enc)/(np.linalg.norm(hip_1)*np.linalg.norm(enc))   # Computing cosine similarity
  sim.append(cos_sim)
proteases['Similarity'] = sim

In [16]:
sorted = proteases.sort_values(by = 'Similarity',ascending = False)
sorted = sorted[['Entry', 'Entry Name', 'Protein names',
       'Organism', 'Length', 'Mass', 'Active site'
      , 'Protein families', 'Encoding',
       'Similarity']]


In [17]:
sorted

,Entry,Entry Name,Protein names,Organism,Length,Mass,Active site,Protein families,Encoding,Similarity
21,P9WHR3,HIP1_MYCTU,Serine protease Hip1 (EC 3.4.21.-) (Hydrolase ...,Mycobacterium tuberculosis (strain ATCC 25618 ...,520,55924,"ACT_SITE 228; /note=""Nucleophile""; /evidence=""...",Peptidase S33 family,"[[-0.055149622, -0.11440211, -0.05563249, 0.01...",1.000000
35,O53945,MYCP5_MYCTU,Mycosin-5 (EC 3.4.21.-) (MycP5 protease),Mycobacterium tuberculosis (strain ATCC 25618 ...,585,60028,"ACT_SITE 109; /note=""Charge relay system""; /ev...",Peptidase S8 family,"[[-0.07025539, -0.21664655, -0.0070105824, 0.0...",0.952266
25,Q8RR56,KSCP_BACX2,Kumamolisin (EC 3.4.21.123) (Kumamolysin) (Kum...,Bacillus sp. (strain MN-32),552,57743,"ACT_SITE 266; /note=""Charge relay system""; /ev...",NaN,"[[-0.17469038, -0.15316765, -0.15177833, 0.028...",0.950980
28,O06291,HTRA1_MYCTU,Probable serine protease HtrA1 (EC 3.4.21.107)...,Mycobacterium tuberculosis (strain ATCC 25618 ...,528,54190,"ACT_SITE 270; /note=""Charge relay system""; /ev...",Peptidase S1C family,"[[-0.17533346, -0.09266855, 0.03147892, -0.007...",0.948543
24,Q8GB88,SCPA_ALISE,Kumamolisin-As (EC 3.4.21.-) (Acid collagenase...,Alicyclobacillus sendaiensis,553,57154,"ACT_SITE 267; /note=""Charge relay system""; /ev...",NaN,"[[-0.13469526, -0.19939965, -0.1564368, 0.0117...",0.948272
17,P42790,PICP_PSESR,Sedolisin (EC 3.4.21.100) (PCP) (Pepstatin-ins...,Pseudomonas sp. (strain 101) (Achromobacter pa...,587,61073,"ACT_SITE 295; /note=""Charge relay system""; /ev...",NaN,"[[-0.13414066, -0.1904276, -0.115057714, 0.046...",0.947860
2,O35002,CTPB_BACSU,Carboxy-terminal processing protease CtpB (C-t...,Bacillus subtilis (strain 168),480,52798,"ACT_SITE 309; /note=""Nucleophile""; /evidence=""...",Peptidase S41A family,"[[-0.13071762, -0.089740716, -0.060651094, -0....",0.944544
0,O04073,CTPA_TETOB,"C-terminal processing peptidase, chloroplastic...",Tetradesmus obliquus (Green alga) (Acutodesmus...,464,48615,"ACT_SITE 372; /note=""Charge relay system""; /ev...",Peptidase S41A family,"[[-0.21536475, -0.034516055, 0.06970948, -0.10...",0.938696
36,P42425,LON2_BACSU,Lon protease 2 (EC 3.4.21.53) (ATP-dependent p...,Bacillus subtilis (strain 168),552,60429,"ACT_SITE 445; /evidence=""ECO:0000255|PROSITE-P...",Peptidase S16 family,"[[-0.083629504, -0.07574105, -0.047649965, -0....",0.936728
39,Q5ZVV9,Q5ZVV9_LEGPH,Protease DO (EC 3.4.21.-),Legionella pneumophila subsp. pneumophila (str...,466,49612,"ACT_SITE 111; /note=""Charge relay system""; /ev...",Peptidase S1C family,"[[-0.18403874, -0.13907455, 0.0010083988, -0.0...",0.935038
